In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

In [2]:
#loading the cleaned data into SQL
orders = pd.read_csv(r"C:\Users\Kiran Kumar\sales-analysis-project\Data\Data-Cleaned\orders_cleaned.csv")
customers = pd.read_csv(r"C:\Users\Kiran Kumar\sales-analysis-project\Data\Data-Cleaned\customers_cleaned.csv")
products = pd.read_csv(r"C:\Users\Kiran Kumar\sales-analysis-project\Data\Data-Cleaned\products_cleaned.csv")
calendar = pd.read_csv(r"C:\Users\Kiran Kumar\sales-analysis-project\Data\Data-Cleaned\calendar_cleaned.csv")

In [3]:
orders.to_sql("orders_cleaned" , conn, index=False, if_exists='replace')
products.to_sql("products_cleaned" , conn, index=False, if_exists='replace')
customers.to_sql("customers_cleaned", conn , index=False , if_exists='replace')
calendar.to_sql("calendar_cleaned", conn, index=False, if_exists="replace")

1096

In [4]:
#Business Question1: What is the total Revenue by Product Category
query = """
SELECT 
    p.category,
    SUM(o.revenue) AS total_revenue
FROM orders_cleaned o
INNER JOIN products_cleaned p
    ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""
pd.read_sql_query(query, conn)

,category,total_revenue
0,Sports,3942935.32
1,Home,3161785.13
2,Office,3139724.29
3,Clothing,2945353.30
4,Electronics,2104313.96


In [5]:
#Business Question2: What is the total Revenue by Region

query = """
SELECT 
    region,
    SUM(revenue) AS total_revenue
FROM orders_cleaned 
GROUP BY region
ORDER by total_revenue DESC
"""
pd.read_sql_query(query, conn)

,region,total_revenue
0,East,3873231.43
1,West,3871728.04
2,South,3796181.84
3,North,3752970.69


In [6]:
#Bussiness Question3: Monthly Revenue Trends (Time-Based Aggregation)

query = """
SELECT 
    c.month,
    c.year,
    SUM(o.revenue) AS total_revenue
FROM orders_cleaned o
INNER JOIN calendar_cleaned c
    ON o.order_date = c.date
GROUP BY c.year, c.month
ORDER BY c.year, c.month
"""

pd.read_sql_query(query, conn)

,month,year,total_revenue
0,1,2022,424619.87
1,2,2022,374257.30
2,3,2022,425126.01
3,4,2022,394769.94
4,5,2022,380524.77
5,6,2022,397044.08
6,7,2022,421121.94
7,8,2022,425276.35
8,9,2022,444301.17
9,10,2022,404436.62


In [7]:
# Business Question 4: Top 10 Products by Total Revenue (Top-N Analysis)

query = """
SELECT 
    p.product_id,
    p.product_name, 
    p.category,
    SUM(o.revenue) AS total_revenue
FROM  products_cleaned p
JOIN  orders_cleaned o 
    ON p.product_id = o.product_id
GROUP BY 
    p.product_id,
    p.product_name, 
    p.category
ORDER BY 
    total_revenue DESC
LIMIT 10
"""

pd.read_sql_query(query, conn)

,product_id,product_name,category,total_revenue
0,21,Product[i],Home,344743.48
1,28,Product[i],Home,341092.69
2,2,Product[i],Office,339857.60
3,22,Product[i],Office,333403.75
4,23,Product[i],Sports,332408.59
5,44,Product[i],Home,330959.73
6,16,Product[i],Home,329087.83
7,38,Product[i],Office,328696.33
8,42,Product[i],Electronics,325278.30
9,13,Product[i],Office,322535.29


In [8]:
query = """
SELECT 
    c.month,
    c.year,
    SUM(o.revenue) AS total_revenue
FROM orders_cleaned o
CROSS JOIN calendar_cleaned c
"""

pd.read_sql_query(query, conn)

,month,year,total_revenue
0,1,2022,1.676235e+10


In [9]:
#Business Question: Top 3 products by revenue within each region (TOP-N)

query = """
WITH product_region_revenue AS(
    SELECT 
        o.region , 
        p.product_name, 
        p.category,
        SUM(o.revenue) AS total_revenue
    FROM 
    orders_cleaned o
    JOIN products_cleaned p 
    ON o.product_id = p.product_id
    GROUP BY o.region, p.product_name, p.category 
),
ranked AS(
    SELECT 
        region,
        product_name,
        category,
        total_revenue,
        RANK() OVER(PARTITION BY region ORDER BY total_revenue DESC) AS revenue_rank
    FROM product_region_revenue
)
SELECT
    region,
    revenue_rank,
    product_name,
    category,
    total_revenue
FROM ranked
WHERE revenue_rank <= 3
ORDER BY region, revenue_rank;
"""
pd.read_sql_query(query, conn)

,region,revenue_rank,product_name,category,total_revenue
0,East,1,Product[i],Sports,991934.92
1,East,2,Product[i],Office,785033.24
2,East,3,Product[i],Home,782722.75
3,North,1,Product[i],Sports,955602.55
4,North,2,Product[i],Home,840301.24
5,North,3,Product[i],Office,765935.16
6,South,1,Product[i],Sports,988696.07
7,South,2,Product[i],Office,784889.74
8,South,3,Product[i],Home,770601.75
9,West,1,Product[i],Sports,1006701.78


In [10]:
#Who are our top customers, and what percentage of total revenue do they contribute (Pareto analysis (80/20 thinking))
#STEP 1: Finding Calculating total revenue per customer

query = """ 
WITH customer_revenue AS(
SELECT 
    c.customer_id,
    c.customer_type,
    c.city,
    SUM(o.revenue) AS total_revenue
FROM 
    customers_cleaned c
JOIN 
    orders_cleaned o 
    ON c.customer_id = o.customer_id
GROUP BY  c.customer_id, c.customer_type, c.city
)
SELECT 
    *
FROM 
    customer_revenue 
ORDER BY total_revenue DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,customer_id,customer_type,city,total_revenue
0,509,B2B,Cologne,32645.10
1,343,B2B,Cologne,31326.20
2,157,B2B,Cologne,28437.29
3,380,B2B,Munich,28340.08
4,570,B2B,Berlin,28034.99
5,499,B2B,Hamburg,27755.35
6,399,B2B,Munich,27700.18
7,898,B2B,Frankfurt,27694.29
8,225,B2B,Cologne,27642.44
9,106,B2B,Cologne,27629.46


In [11]:
# STEP 2: Adding total % Contribution
query = """ 
WITH customer_revenue AS (
    SELECT
        c.customer_id,
        c.customer_type,
        c.city,
        SUM(o.revenue) AS total_revenue
    FROM orders_cleaned o
    INNER JOIN customers_cleaned c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_type, c.city
)
SELECT
    customer_id,
    customer_type,
    city,
    total_revenue,
    ROUND(
        total_revenue * 100.0 / SUM(total_revenue) OVER (),
        2
    ) AS revenue_share_percent
FROM customer_revenue
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_id,customer_type,city,total_revenue,revenue_share_percent
0,509,B2B,Cologne,32645.10,0.21
1,343,B2B,Cologne,31326.20,0.20
2,157,B2B,Cologne,28437.29,0.19
3,380,B2B,Munich,28340.08,0.19
4,570,B2B,Berlin,28034.99,0.18
5,499,B2B,Hamburg,27755.35,0.18
6,399,B2B,Munich,27700.18,0.18
7,898,B2B,Frankfurt,27694.29,0.18
8,225,B2B,Cologne,27642.44,0.18
9,106,B2B,Cologne,27629.46,0.18


In [12]:
monthly_revenue = pd.read_sql_query("""
SELECT
    c.year,
    c.month,
    SUM(o.revenue) AS total_revenue
FROM orders_cleaned o
INNER JOIN calendar_cleaned c
    ON o.order_date = c.date
GROUP BY c.year, c.month
ORDER BY c.year, c.month;
""", conn)

monthly_revenue.to_excel(r"C:\Users\Kiran Kumar\sales-analysis-project\Excel\monthly_revenue.xlsx", index=False)

In [14]:
category_revenue = pd.read_sql_query("""
SELECT
    p.category,
    SUM(o.revenue) AS total_revenue
FROM orders_cleaned o
INNER JOIN products_cleaned p
    ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
""", conn)

category_revenue.to_excel(r"C:\Users\Kiran Kumar\sales-analysis-project\Excel\category_revenue.xlsx", index=False)

In [15]:
region_revenue = pd.read_sql_query("""
SELECT
    region,
    SUM(revenue) AS total_revenue
FROM orders_cleaned
GROUP BY region
ORDER BY total_revenue DESC;
""", conn)

region_revenue.to_excel(r"C:\Users\Kiran Kumar\sales-analysis-project\Excel\region_revenue.xlsx", index=False)

In [17]:
top10_products = pd.read_sql_query("""
SELECT
    p.product_name,
    p.category,
    COUNT(o.order_id) AS total_orders,
    SUM(o.revenue) AS total_revenue
FROM orders_cleaned o
INNER JOIN products_cleaned p
    ON o.product_id = p.product_id
GROUP BY p.product_name, p.category
ORDER BY total_revenue DESC
LIMIT 10;
""", conn)
top10_products.to_excel(r"C:\Users\Kiran Kumar\sales-analysis-project\Excel\top10_products.xlsx", index=False)